In [ ]:
# 1. Sistem Araçları (COLMAP, FFmpeg, Sanal Ekran, ZBar)
!sudo apt-get update -y
!sudo apt-get install -y ffmpeg xvfb libzbar0

In [ ]:
!pip install /kaggle/input/datasets/yusuftiryaki/wheels/*.whl

In [ ]:
!pip install firebase-admin pyzbar opencv-python open3d plyfile

In [ ]:
# Balyoz 2.0: Dosyayı Python ile açıp içindeki tüm Webrtc tetikleyicilerini siliyoruz
file_path = '/usr/local/lib/python3.12/dist-packages/open3d/__init__.py'

with open(file_path, 'r') as file:
    lines = file.readlines()

with open(file_path, 'w') as file:
    for line in lines:
        # Eğer satırda bu bozuk modül çağrılıyorsa, o satırı 'pass' ile değiştir!
        if 'open3d.visualization.webrtc_server' in line:
            file.write('                pass\n')
        else:
            file.write(line)

print("Open3D'nin bozuk Jupyter başlatıcısı tamamen temizlendi!")

In [ ]:
import os
import cv2
import numpy as np
import open3d as o3d
from pyzbar.pyzbar import decode
import firebase_admin
from firebase_admin import credentials, firestore, storage
import subprocess
import time
import glob
import urllib.parse
import json
from plyfile import PlyData

In [ ]:
# 4. Nerfstudio Kurulumu
!pip install nerfstudio

In [ ]:
def analyze_video_for_hybrid_plate(video_path):
    """
    Videoyu tarar:
    1. QR koddan karmaşık Ağaç ID'sini okur (Veri).
    2. ArUco Marker'dan piksel genişliğini ölçer (Geometri/Cetvel).
    """
    print(f"[DEBUG] Hibrit plaka analizi başlıyor: {video_path}")
    cap = cv2.VideoCapture(video_path)

    # ArUco ayarları (4x4 sözlüğü - tarla için en okunaklısı)
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    parameters = cv2.aruco.DetectorParameters()
    detector = cv2.aruco.ArucoDetector(aruco_dict, parameters)

    tree_id = None
    marker_pixel_width = None
    frame_count = 0

    # İlk 300 karede (yaklaşık ilk 10 saniye) plakayı bulmaya çalışıyoruz
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_count > 300:
            break

        # İşlemeyi hızlandırmak için kareyi siyah/beyaza çevir
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # ---------------------------------------------------------
        # 1. GÖREV: QR KOD TARAMASI (Ağaç ID'si)
        # ---------------------------------------------------------
        if tree_id is None:
            decoded_objects = decode(gray)
            for obj in decoded_objects:
                tree_id = obj.data.decode('utf-8')
                print(f"[DEBUG] Frame {frame_count}'te QR Kod Okundu! Ağaç ID: {tree_id}")
                break # İlk bulduğunu al ve döngüden çık

        # ---------------------------------------------------------
        # 2. GÖREV: ARUCO TARAMASI (Geometrik Cetvel)
        # ---------------------------------------------------------
        if marker_pixel_width is None:
            corners, ids, rejectedImgPoints = detector.detectMarkers(gray)
            if ids is not None:
                # İlk ArUco'nun köşeleri: [Sol Üst, Sağ Üst, Sağ Alt, Sol Alt]
                c = corners[0][0]
                # Sol Üst ile Sağ Üst köşe arasındaki piksel mesafesini hesapla
                marker_pixel_width = np.linalg.norm(c[0] - c[1])
                print(f"[DEBUG] Frame {frame_count}'te ArUco Marker Bulundu!")
                print(f"[DEBUG] ArUco Piksel Genişliği: {marker_pixel_width:.2f}")

        # ---------------------------------------------------------
        # ÇIKIŞ KOŞULU: İkisi de bulunduysa videoyu okumayı bırak (Zaman tasarrufu)
        # ---------------------------------------------------------
        if tree_id is not None and marker_pixel_width is not None:
            print(f"[DEBUG] Hibrit plaka başarıyla çözüldü. Video analizi durduruluyor.")
            break

        frame_count += 1

    cap.release()
    return tree_id, marker_pixel_width, frame_count

In [51]:
import numpy as np
from plyfile import PlyData

def convert_ply_to_splat(ply_file_path, splat_file_path):
    """
    Nerfstudio'nun ürettiği ağır .ply dosyasını okur, matematiksel dönüşümleri
    yapar ve web için optimize edilmiş .splat formatına çevirir.
    WebGL çökmelerini engellemek için NaN korumaları, OUTLIER FİLTRESİ 
    ve özellik okuma (renk/scale) düzeltmeleri eklenmiştir.
    """
    print(f"[DEBUG] Web optimizasyonu başlıyor. .ply -> .splat formatına sıkıştırılıyor...")

    # 1. PLY verisini oku
    plydata = PlyData.read(ply_file_path)
    vertex = plydata['vertex']
    num_pts = len(vertex['x'])
    
    # 🚨 DÜZELTME: Objeleri metin (string) listesine çeviriyoruz ki if sorguları çalışsın!
    property_names = [p.name for p in vertex.properties]

    # 2. Koordinatlar (X, Y, Z) - ÇÖKME ÖNLEMİ: NaN veya Inf değerlerini temizle
    x_raw = np.nan_to_num(np.array(vertex['x']), nan=0.0, posinf=0.0, neginf=0.0)
    y_raw = np.nan_to_num(np.array(vertex['y']), nan=0.0, posinf=0.0, neginf=0.0)
    z_raw = np.nan_to_num(np.array(vertex['z']), nan=0.0, posinf=0.0, neginf=0.0)

    # -------------------------------------------------------------
    # SERSERİ NOKTA (OUTLIER) FİLTRESİ
    # Merkeze 15 metreden daha uzak olan gürültü piksellerini çöpe at
    # Bu sayede WebGL kamerasının patlaması kesin olarak engellenir.
    # -------------------------------------------------------------
    distance_limit = 15.0
    valid_mask = (np.abs(x_raw) < distance_limit) & (np.abs(y_raw) < distance_limit) & (np.abs(z_raw) < distance_limit)
    
    # Sadece geçerli, merkeze yakın noktaları alıyoruz
    x = x_raw[valid_mask]
    y = y_raw[valid_mask]
    z = z_raw[valid_mask]

    valid_pts = len(x)
    print(f"[DEBUG] Uzaya fırlamış {num_pts - valid_pts} adet bozuk nokta silindi. Kalan Sağlam Splat: {valid_pts}")

    # 3. Renkler (Spherical Harmonics'ten RGB'ye çevrim)
    SH_C0 = 0.28209479177387814
    if 'f_dc_0' in property_names:
        r = np.clip((np.nan_to_num(np.array(vertex['f_dc_0']))[valid_mask] * SH_C0 + 0.5), 0.0, 1.0) * 255.0
        g = np.clip((np.nan_to_num(np.array(vertex['f_dc_1']))[valid_mask] * SH_C0 + 0.5), 0.0, 1.0) * 255.0
        b = np.clip((np.nan_to_num(np.array(vertex['f_dc_2']))[valid_mask] * SH_C0 + 0.5), 0.0, 1.0) * 255.0
    elif 'red' in property_names:
        r = np.clip(np.nan_to_num(np.array(vertex['red']))[valid_mask], 0, 255)
        g = np.clip(np.nan_to_num(np.array(vertex['green']))[valid_mask], 0, 255)
        b = np.clip(np.nan_to_num(np.array(vertex['blue']))[valid_mask], 0, 255)
    else:
        print("⚠️ HATA: PLY dosyasında renk özelliği bulunamadı!")
        r, g, b = np.zeros(valid_pts), np.zeros(valid_pts), np.zeros(valid_pts)

    # 4. Şeffaflık / Opacity (Sigmoid)
    if 'opacity' in property_names:
        opacity_raw = np.nan_to_num(np.array(vertex['opacity']))[valid_mask]
        opacity = 1.0 / (1.0 + np.exp(-opacity_raw))
        a = opacity * 255.0
    else:
        a = np.ones(valid_pts) * 255.0

    # 5. Boyutlar / Scale
    if 'scale_0' in property_names:
        s0 = np.clip(np.nan_to_num(np.array(vertex['scale_0']))[valid_mask], -10.0, 5.0)
        s1 = np.clip(np.nan_to_num(np.array(vertex['scale_1']))[valid_mask], -10.0, 5.0)
        s2 = np.clip(np.nan_to_num(np.array(vertex['scale_2']))[valid_mask], -10.0, 5.0)
        sx, sy, sz = np.exp(s0), np.exp(s1), np.exp(s2)
    else:
        # Piyasada scale verisi dönmeyen PLY dosyaları için varsayılan boyut
        sx, sy, sz = np.ones(valid_pts)*0.01, np.ones(valid_pts)*0.01, np.ones(valid_pts)*0.01

    # 6. Dönüş / Rotation (Quaternion normalizasyonu)
    if 'rot_0' in property_names:
        rw = np.nan_to_num(np.array(vertex['rot_0']))[valid_mask]
        rx = np.nan_to_num(np.array(vertex['rot_1']))[valid_mask]
        ry = np.nan_to_num(np.array(vertex['rot_2']))[valid_mask]
        rz = np.nan_to_num(np.array(vertex['rot_3']))[valid_mask]

        norms = np.sqrt(rw**2 + rx**2 + ry**2 + rz**2)
        norms[norms == 0] = 1e-6 
        
        rw, rx, ry, rz = rw/norms, rx/norms, ry/norms, rz/norms

        qw = np.clip((rw * 128) + 128, 0, 255)
        qx = np.clip((rx * 128) + 128, 0, 255)
        qy = np.clip((ry * 128) + 128, 0, 255)
        qz = np.clip((rz * 128) + 128, 0, 255)
    else:
        qw, qx, qy, qz = np.ones(valid_pts)*255, np.zeros(valid_pts), np.zeros(valid_pts), np.zeros(valid_pts)

    # 7. BINARY SIKIŞTIRMA (Little-Endian / < işareti)
    dt = np.dtype([
        ('x', '<f4'), ('y', '<f4'), ('z', '<f4'),
        ('sx', '<f4'), ('sy', '<f4'), ('sz', '<f4'),
        ('r', 'u1'), ('g', 'u1'), ('b', 'u1'), ('a', 'u1'),
        ('qx', 'u1'), ('qy', 'u1'), ('qz', 'u1'), ('qw', 'u1')
    ])

    splat_data = np.zeros(valid_pts, dtype=dt)
    splat_data['x'], splat_data['y'], splat_data['z'] = x, y, z
    splat_data['sx'], splat_data['sy'], splat_data['sz'] = sx, sy, sz
    
    splat_data['r'] = r.astype(np.uint8)
    splat_data['g'] = g.astype(np.uint8)
    splat_data['b'] = b.astype(np.uint8)
    splat_data['a'] = a.astype(np.uint8)
    
    splat_data['qx'] = qx.astype(np.uint8)
    splat_data['qy'] = qy.astype(np.uint8)
    splat_data['qz'] = qz.astype(np.uint8)
    splat_data['qw'] = qw.astype(np.uint8)

    with open(splat_file_path, "wb") as f:
        f.write(splat_data.tobytes())

    print(f"[SONUÇ] Mükemmel! Renkli ve filtrelenmiş model .splat formatına başarıyla dönüştürüldü: {splat_file_path}")
    return splat_file_path

In [ ]:
def calculate_fractal_dimension(pcd_crown):
    """
    Sadece yaprak ve dalları içeren taç kısmının (Nokta Bulutu)
    3D Kutu Sayma (Box-Counting) yöntemiyle fraktal skorunu hesaplar.
    """
    print("[DEBUG] Taç apısının Fraktal Skoru hesaplanıyor...")

    # Noktaları numpy dizisine çevir
    points = np.asarray(pcd_crown.points)

    if len(points) == 0:
        return 0.0

    # 1. Noktaları Normalize Et (Tüm modeli 0 ile 1 arasında bir küpün içine sıkıştır)
    points_min = points.min(axis=0)
    points_max = points.max(axis=0)
    points_normalized = (points - points_min) / (points_max - points_min)

    # 2. Kutu boyutlarını (Ölçekleri) belirle
    # 2^1'den 2^8'e kadar logaritmik olarak artan ızgara (grid) çözünürlükleri
    scales = np.logspace(1, 8, num=8, endpoint=True, base=2)

    Ns = [] # Dolu kutu sayılarını tutacağımız liste

    for scale in scales:
        # Noktaların o anki ızgara (grid) içindeki koordinatlarını tam sayıya yuvarlayarak bul
        grid_indices = np.floor(points_normalized * scale).astype(int)

        # Benzersiz (içinde en az 1 nokta olan) kutuları say
        # axis=0 ile [x, y, z] koordinat üçlülerinin benzersiz olanlarını buluyoruz
        unique_boxes = len(np.unique(grid_indices, axis=0))
        Ns.append(unique_boxes)

    # 3. Log-Log Grafiğinin Eğimini Hesapla (Lineer Regresyon / Polyfit)
    # x ekseni: log(scale) -> Kutu sayısının tersi (1/epsilon)
    # y ekseni: log(Ns) -> Dolu kutu sayısı
    coeffs = np.polyfit(np.log(scales), np.log(Ns), 1)

    fractal_score = coeffs[0] # Doğrunun eğimi bize doğrudan Fraktal Boyutu (D_f) verir

    print(f"[SONUÇ] Ağacın Fraktal Skoru (D_f): {fractal_score:.3f}")

    return fractal_score

In [ ]:
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist

def calculate_additional_metrics(pcd, height):
    """
    3D nokta bulutundan ek ağaç metriklerini hesaplar.
    pcd: Zaten ölçeklenmiş ve Z=0'a oturtulmuş Open3D nokta bulutu.
    height: Daha önce hesaplanmış ağaç boyu (metre).
    
    Döndürülen Metrikler:
    - Taç Çapı (m)
    - Taç Yüzey Alanı (m²)
    - Gövde Eğikliği (derece)
    - Taç Asimetri İndeksi (0-1)
    - Taç Yoğunluğu (nokta/m³)
    - Taç Tabanı Yüksekliği (m)
    """
    points = np.asarray(pcd.points)

    # ------------------------------------------------------------------
    # 1. TAÇ ÇAPI (Crown Diameter) — Kuş bakışı XY düzleminde max genişlik
    # ------------------------------------------------------------------
    crown_mask = points[:, 2] > 0.5  # Gövdeyi hariç tut
    crown_points = points[crown_mask]

    crown_diameter_m = 0.0
    if len(crown_points) > 10:
        crown_xy = crown_points[:, :2]
        # Bounding Box ile ilk tahmin
        xy_min = crown_xy.min(axis=0)
        xy_max = crown_xy.max(axis=0)
        crown_diameter_m = float(np.max(xy_max - xy_min))

        # Daha hassas: 2D Convex Hull köşeleri arasındaki max mesafe
        try:
            hull_2d = ConvexHull(crown_xy)
            hull_vertices = crown_xy[hull_2d.vertices]
            crown_diameter_m = float(np.max(pdist(hull_vertices)))
        except Exception:
            pass  # Yeterli nokta yoksa bounding box sonucunu kullan

    print(f"[SONUÇ] Taç Çapı: {crown_diameter_m:.2f} m")

    # ------------------------------------------------------------------
    # 2. TAÇ YÜZEY ALANI (Crown Surface Area — Convex Hull)
    # ------------------------------------------------------------------
    crown_surface_area_m2 = 0.0
    crown_pcd = pcd.select_by_index(np.where(crown_mask)[0])
    try:
        hull_mesh, _ = crown_pcd.compute_convex_hull()
        crown_surface_area_m2 = float(hull_mesh.get_surface_area())
    except Exception:
        pass

    print(f"[SONUÇ] Taç Yüzey Alanı: {crown_surface_area_m2:.2f} m²")

    # ------------------------------------------------------------------
    # 3. GÖVDE EĞİKLİĞİ (Trunk Lean Angle)
    # Gövde noktalarına (10-80cm arası) PCA ile ana eksen bulup dikeyden sapmasını ölç
    # ------------------------------------------------------------------
    trunk_mask = (points[:, 2] > 0.1) & (points[:, 2] < 0.8)
    trunk_points = points[trunk_mask]
    trunk_lean_deg = 0.0

    if len(trunk_points) > 20:
        trunk_center = trunk_points.mean(axis=0)
        trunk_centered = trunk_points - trunk_center
        cov_matrix = np.cov(trunk_centered.T)
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        # En büyük eigenvalue'nun vektörü = gövdenin ana yönü
        principal_axis = eigenvectors[:, np.argmax(eigenvalues)]

        # Dikey vektör (0, 0, 1) ile arasındaki açı
        cos_angle = abs(np.dot(principal_axis, [0, 0, 1])) / np.linalg.norm(principal_axis)
        trunk_lean_deg = float(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))

    print(f"[SONUÇ] Gövde Eğikliği: {trunk_lean_deg:.1f}°")

    # ------------------------------------------------------------------
    # 4. TAÇ ASİMETRİSİ (Crown Asymmetry Index)
    # Taç merkezi ile gövde merkezinin XY düzlemindeki sapma oranı
    # 0 = tamamen simetrik, 1'e yakın = çok asimetrik
    # ------------------------------------------------------------------
    asymmetry_index = 0.0
    if len(crown_points) > 10 and len(trunk_points) > 10:
        crown_center_xy = crown_points[:, :2].mean(axis=0)
        trunk_center_xy = trunk_points[:, :2].mean(axis=0)
        asymmetry_m = float(np.linalg.norm(crown_center_xy - trunk_center_xy))
        asymmetry_index = asymmetry_m / crown_diameter_m if crown_diameter_m > 0 else 0.0

    print(f"[SONUÇ] Taç Asimetri İndeksi: {asymmetry_index:.3f}")

    # ------------------------------------------------------------------
    # 5. TAÇ YOĞUNLUĞU (Crown Density — nokta/m³)
    # ------------------------------------------------------------------
    crown_density = 0.0
    try:
        hull_mesh_vol, _ = crown_pcd.compute_convex_hull()
        crown_vol = hull_mesh_vol.get_volume()
        crown_density = float(len(crown_points) / crown_vol) if crown_vol > 0 else 0.0
    except Exception:
        pass

    print(f"[SONUÇ] Taç Yoğunluğu: {crown_density:.0f} nokta/m³")

    # ------------------------------------------------------------------
    # 6. TAÇ TABANI YÜKSEKLİĞİ (Crown Base Height)
    # Yapraklanmanın yoğun olarak başladığı Z seviyesi
    # ------------------------------------------------------------------
    z_bins = np.linspace(0, height, num=50)
    z_counts = np.histogram(points[:, 2], bins=z_bins)[0]

    # Toplam noktaların %5'ini aşan ilk yükseklik dilimi = taç tabanı
    threshold = len(points) * 0.05 / len(z_bins)
    crown_base_height = 0.0
    for i in range(len(z_counts)):
        if z_counts[i] > threshold and z_bins[i] > 0.3:  # 30cm üstü (gövdeyi atla)
            crown_base_height = float(z_bins[i])
            break

    print(f"[SONUÇ] Taç Tabanı Yüksekliği: {crown_base_height:.2f} m")

    return {
        "crown_diameter_m": round(crown_diameter_m, 2),
        "crown_surface_area_m2": round(crown_surface_area_m2, 2),
        "trunk_lean_deg": round(trunk_lean_deg, 1),
        "crown_asymmetry_index": round(asymmetry_index, 3),
        "crown_density_pts_m3": round(crown_density, 0),
        "crown_base_height_m": round(crown_base_height, 2),
    }

In [ ]:
def calculate_scale_factor(transforms_json_path, frame_count, pixel_width, real_width_m=0.15):
    """
    Kamera matematiğini kullanarak 3D model ile gerçek dünya arasındaki ölçek çarpanını bulur.
    """
    print("[DEBUG] Ölçek Çarpanı (Scale Factor) hesaplanıyor...")

    # 1. COLMAP'in oluşturduğu kamera verilerini oku
    with open(transforms_json_path, 'r') as f:
        data = json.load(f)

    # 2. Odak uzaklığını (Focal Length) al (COLMAP'in kusursuz hesapladığı değer)
    fl_x = data["fl_x"]

    # 3. GERÇEK DÜNYA MESAFESİ (Kamera -> Ağaç Gövdesi)
    real_distance_m = (fl_x * real_width_m) / pixel_width
    print(f"[DEBUG] Kameranın ağaca GERÇEK mesafesi: {real_distance_m:.2f} metre")

    # 4. SANAL DÜNYA MESAFESİ (COLMAP'in evrenindeki mesafe)
    # FFmpeg kareleri genelde frame_00001.jpg gibi kaydeder, formatı ayarlıyoruz
    # (frame_count + 1 yapıyoruz çünkü index 0'dan başlıyor, dosya isimleri 1'den)
    target_frame_name = f"frame_{frame_count + 1:05d}"

    virtual_distance = None
    for frame in data["frames"]:
        if target_frame_name in frame["file_path"]:
            # transform_matrix 4x4 bir matristir. X, Y, Z konumu sağ üsttedir.
            matrix = np.array(frame["transform_matrix"])
            camera_x = matrix[0, 3]
            camera_y = matrix[1, 3]
            camera_z = matrix[2, 3]

            # Kameranın merkeze (0,0,0) olan 3D uzaklığını hesapla (Öklid Mesafesi)
            virtual_distance = math.sqrt(camera_x**2 + camera_y**2 + camera_z**2)
            print(f"[DEBUG] Kameranın ağaca SANAL mesafesi: {virtual_distance:.4f} birim")
            break

    if virtual_distance is None:
        print("UYARI: ArUco bulunan kare transforms.json içinde bulunamadı! Varsayılan çarpan 1.0 kullanılıyor.")
        return 1.0

    # 5. ALTIN ÇARPAN (SCALE FACTOR)
    scale_factor = real_distance_m / virtual_distance
    print(f"[SONUÇ] Hesaplanan Ölçek Çarpanı: {scale_factor:.4f}")

    return scale_factor

In [ ]:
def measure_tree_metrics(ply_file_path, scale_factor):
    """
    3D nokta bulutunu alır, ArUco referansına göre metrik (metre) sisteme çevirir,
    toprağı bulur ve ağacın boyunu/hacmini hesaplar.
    Ek olarak taç çapı, yüzey alanı, gövde eğikliği, asimetri, yoğunluk ve taç tabanı yüksekliğini hesaplar.
    """
    print(f"[DEBUG] 3D Model analiz ediliyor: {ply_file_path}")

    # 1. Nokta Bulutunu Yükle
    pcd = o3d.io.read_point_cloud(ply_file_path)

    # 2. Gürültü Temizleme (Havada asılı kalan hatalı pikselleri sil)
    # nb_neighbors: Her noktanın etrafındaki komşu sayısı, std_ratio: Standart sapma toleransı
    cl, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    pcd = pcd.select_by_index(ind)

    # 3. ÖLÇEKLEME (Scaling) - Cetveli Vurmak
    # Katsayı = Gerçek Boyut (metre) / COLMAP'in Sanal Boyutu
    print(f"[DEBUG] Uygulanan Ölçek Çarpanı: {scale_factor:.4f}")

    # Tüm modeli ölçeklendir (Artık model metre cinsinden)
    pcd.scale(scale_factor, center=pcd.get_center())

    # 4. ZEMİN TESPİTİ (Toprağı Bulmak)
    # Noktaların Z eksenini (yukarı/aşağı) al. En alttaki %1'lik dilimi zemin kabul et.
    # (%1 kullanıyoruz çünkü tek tük aşağı sarkan gürültü noktalarına aldanmak istemeyiz)
    points = np.asarray(pcd.points)
    z_coords = points[:, 2] # 2. indeks Z eksenidir
    ground_z = np.percentile(z_coords, 1.0)

    # Tüm ağacı aşağı/yukarı kaydırarak toprağı tam Z = 0 noktasına oturt
    translation_vector = np.array([0, 0, -ground_z])
    pcd.translate(translation_vector)

    # 5. AĞAÇ BOYUNU HESAPLAMA
    # Toprak artık Z=0 olduğuna göre, en yüksek Z noktası bize boyu verir
    points_shifted = np.asarray(pcd.points)
    tree_height_m = np.max(points_shifted[:, 2])

    # 6. TAÇ HACMİ HESAPLAMA (Convex Hull)
    # Gövdeyi hesaba katmamak için topraktan itibaren ilk 50 cm'yi (0.5m) kırpıyoruz
    crown_indices = np.where(points_shifted[:, 2] > 0.5)[0]
    crown_pcd = pcd.select_by_index(crown_indices)

    # Kalan yapraklı kısmın etrafını sanal bir zarla sar (Convex Hull) ve hacmini ölç
    hull, _ = crown_pcd.compute_convex_hull()
    crown_volume_m3 = hull.get_volume()

    # 7. GÖVDE ÇAPI HESAPLAMA (Trunk Diameter)
    # Fıstık ağaçları için topraktan 30 cm ile 50 cm (0.3m - 0.5m) arasındaki net gövde dilimini alıyoruz
    trunk_indices = np.where((points_shifted[:, 2] > 0.3) & (points_shifted[:, 2] < 0.5))[0]
    trunk_pcd = pcd.select_by_index(trunk_indices)

    # Bu dilimdeki noktaların sadece X ve Y koordinatlarını al (Kuş bakışı 2D görünüme geç)
    trunk_points_2d = np.asarray(trunk_pcd.points)[:, :2]

    if len(trunk_points_2d) > 10: # Eğer o yükseklikte yeterli gövde noktası varsa
        # 1. Gövdenin tam merkez noktasını (ortalama X ve Y) bul
        center_2d = np.mean(trunk_points_2d, axis=0)

        # 2. Merkezden gövdenin dış kabuğundaki tüm noktalara olan mesafeleri ölç
        distances = np.linalg.norm(trunk_points_2d - center_2d, axis=1)

        # 3. Yarıçapı bul. (Kamera gürültüsünden/çöplerden etkilenmemek için median -ortanca- değeri alıyoruz)
        radius_m = np.median(distances)

        # 4. Yarıçapı 2 ile çarpıp çapa ulaş, ardından santimetreye çevir
        trunk_diameter_cm = (radius_m * 2) * 100

        print(f"[SONUÇ] Gövde Çapı (30-50cm arası): {trunk_diameter_cm:.1f} cm")
    else:
        trunk_diameter_cm = 0.0
        print("[UYARI] Gövde çapı ölçülemedi (O yükseklikte yeterli nokta bulunamadı).")

    # 8. Fraktal Boyut hesaplama
    crown_indices = np.where(points_shifted[:, 2] > 0.5)[0]
    crown_pcd = pcd.select_by_index(crown_indices)
    fractal_score = calculate_fractal_dimension(crown_pcd)

    # 9. EK METRİKLER (Taç Çapı, Yüzey Alanı, Gövde Eğikliği, Asimetri, Yoğunluk, Taç Tabanı)
    print("[DEBUG] Ek metrikler hesaplanıyor...")
    additional_metrics = calculate_additional_metrics(pcd, tree_height_m)

    print(f"[SONUÇ] Ağaç Boyu: {tree_height_m:.2f} Metre")
    print(f"[SONUÇ] Taç Hacmi: {crown_volume_m3:.2f} Metreküp")
    print(f"[SONUÇ] Ağaç Gövde Çapı: {trunk_diameter_cm:.2f} Santimetre")
    print(f"[SONUÇ] Ağaç Fraktal Boyutu: {fractal_score:.2f} Skalar")

    return tree_height_m, crown_volume_m3, trunk_diameter_cm, fractal_score, ground_z, pcd, additional_metrics

In [ ]:
# --- 1. FIREBASE BAĞLANTISI ---
# Firebase konsolundan indirdiğiniz serviceAccountKey.json dosyasını Colab'a yükleyin
CREDENTIALS_PATH = "/kaggle/input/datasets/yusuftiryaki/firabase-keys/firebase.json"
STORAGE_BUCKET = "studio-166104040-52130.firebasestorage.app" # Kendi bucket isminizle değiştirin

if not firebase_admin._apps:
    cred = credentials.Certificate(CREDENTIALS_PATH)
    firebase_admin.initialize_app(cred, {
        'storageBucket': STORAGE_BUCKET
    })

db = firestore.client()
bucket = storage.bucket()

# Sistem Sabitleri
QR_GERCEK_BOYUT_METRE = 0.15  # 15 cm x 15 cm QR kod kullandığımızı varsayıyoruz
WORKSPACE_DIR = "/kaggle/working/pistachio_workspace"
os.makedirs(WORKSPACE_DIR, exist_ok=True)


In [ ]:
def run_colmap_processing(video_path, doc_id, colmap_dir):
    """FFmpeg kullanarak videodan kareler çıkarır ve COLMAP ile kamera pozisyonlarını hesaplar."""
    print(f"[{doc_id}] Adım 1/3: FFmpeg ve COLMAP çalışıyor (Bu adım videonun uzunluğuna göre 5-15 dk sürer)...")

    colmap_db_path = os.path.join(colmap_dir, "colmap", "database.db")
    if os.path.exists(colmap_db_path):
        print(f"[DEBUG] COLMAP veritabanı {colmap_db_path} zaten mevcut. Adım 1 atlanıyor.")
        return

    try:
        cmd = [
            "xvfb-run", "-a", # Sanal ekran tetikleyici eklendi!
            "ns-process-data", "video",
            "--data", video_path,
            "--output-dir", colmap_dir,
            "--num-frames-target", "60",         # Test için 60 kareye indirdik (hızlı bitsin)
            "--matching-method", "sequential",   # Videolar için işlemciyi kurtaran sihirli ayar
            "--verbose"
        ]
        print(f"[DEBUG] Running command: {' '.join(cmd)}")

        # Qt platform ve OpenGL hatalarını çözmek için ortam değişkenlerini ayarla
        env = os.environ.copy()
        env["QT_QPA_PLATFORM"] = "offscreen"
        
        # Sadece bu satır aktif kalacak, diğerini siliyoruz!
        result = subprocess.run(cmd, check=True, capture_output=True, text=True, env=env)
        
        print("ns-process-data stdout:\n", result.stdout)
        print("ns-process-data stderr:\n", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"ns-process-data command failed. Exit code: {e.returncode}")
        print(f"ns-process-data stdout on error:\n{e.stdout}")
        print(f"ns-process-data stderr on error:\n{e.stderr}")
        raise Exception(f"ns-process-data komutu başarısız oldu: {e}")

In [ ]:
def run_3d_reconstruction_pipeline(video_path, doc_id):
    print(f"[{doc_id}] GERÇEK 3D ÜRETİM BAŞLIYOR...")

    # Her ağaç için izole bir çalışma alanı yarat
    base_dir = f"{WORKSPACE_DIR}/{doc_id}"
    colmap_dir = f"{base_dir}/colmap_data"
    export_dir = f"{base_dir}/export"
    os.makedirs(colmap_dir, exist_ok=True)
    os.makedirs(export_dir, exist_ok=True)


    # ADIM 1: FFmpeg ve COLMAP (Kareleri çıkar ve kamera pozisyonlarını hesapla)
    run_colmap_processing(video_path, doc_id, colmap_dir)

    # ADIM 2: 3D Gaussian Splatting (Splatfacto) Eğitimi
    train_splatfacto_model(doc_id=doc_id, colmap_dir=colmap_dir)

    # ADIM 3: Eğitilen Modeli .PLY formatında dışa aktarma
    print("Adım 3/3: Model .ply formatında dışa aktarılıyor...")

    # Nerfstudio eğitilen modelin config dosyasını dinamik bir klasöre atar. Onu bulmamız lazım.
    # Yol formatı: outputs/<colmap_klasörü_adı>/splatfacto/<tarih_saat>/config.yml
    colmap_folder_name = os.path.basename(colmap_dir)
    config_search_path = f"{base_dir}/outputs/colmap_data/splatfacto/*/config.yml"

    try:
        config_path = glob.glob(config_search_path)[0]
    except IndexError:
        raise Exception("Eğitim tamamlandı ama config.yml bulunamadı. Model çöküp üretilememiş olabilir.")

    # ns-export komutu gaussian splatting modelini ölçüm yapabileceğimiz .ply dosyasına çevirir.
    subprocess.run(
        [
            "ns-export", "gaussian-splat",
            "--load-config", config_path,
            "--output-dir", export_dir
        ], check=True, cwd=base_dir
    )

    # ns-export komutu varsayılan olarak dosyayı 'splat.ply' adıyla kaydeder
    final_ply_path = os.path.join(export_dir, "splat.ply")

    print(f"[{doc_id}] GERÇEK 3D ÜRETİM TAMAMLANDI. Çıktı: {final_ply_path}")

    # Bu ply dosyası artık Open3D'ye (önceki koddaki scale_and_measure_tree fonksiyonuna) gidiyor.
    return final_ply_path

In [ ]:
def train_splatfacto_model(doc_id, colmap_dir):
    """3D Gaussian Splatting (Splatfacto) modelini eğitir."""
    print(f"[{doc_id}] Adım 2/3: 3DGS modeli eğitiliyor (GPU tam kapasite devrede, 15-25 dk sürer)....")

    # Eğitilmiş modelin config.yml dosyasının varlığını kontrol et
    colmap_folder_name = os.path.basename(colmap_dir)
    config_search_path = f"{WORKSPACE_DIR}/{doc_id}/outputs/colmap_data/splatfacto/*/config.yml"
    existing_configs = glob.glob(config_search_path)

    if existing_configs:
        print(f"[DEBUG] Eğitilmiş model {existing_configs[0]} zaten mevcut. Adım 2 atlanıyor.")
        return

    try:
      custom_env = os.environ.copy()
      custom_env["TORCH_COMPILE_DISABLE"] = "1"
      custom_env["MAX_JOBS"] = "2"
      abs_colmap_dir = os.path.abspath(colmap_dir)
      splat_result = subprocess.run([
          "ns-train", "splatfacto",
          "--data", abs_colmap_dir,
          "--max-num-iterations", "7000",
          "--vis", "tensorboard" # ÇÖZÜM 1: Canlı izleme arayüzünü tamamen kapatır, port çökmesini önler.
      ], check=True, capture_output=True, text=True, env=custom_env, cwd=f"{WORKSPACE_DIR}/{doc_id}")

      print("ns-train stdout:\n", splat_result.stdout)
      print("ns-train stderr:\n", splat_result.stderr)

    except subprocess.CalledProcessError as e:
      print(f"ns-train command failed. Exit code: {e.returncode}")
      print(f"ns-train stdout on error:\n{e.stdout}")
      print(f"ns-train stderr on error:\n{e.stderr}")
      raise Exception(f"ns-train komutu başarısız oldu: {e}")

In [ ]:
import uuid
import urllib.parse
from firebase_admin import firestore, storage

def upload_metrics_and_update_db(local_image_path, splat_model_path, ply_model_path, doc_id, height, diameter, volume, fractal_score, scale_factor, ground_z, qr_code="unknown", additional_metrics=None):
    """
    Oluşturulan 3D ve 2D metrik görüntüsünü Firebase Storage'a yükler,
    KALICI Firebase Download Token'ı üreterek PWA için public URL alır ve Firestore'a kaydeder.
    PLY formatındaki ham modeli de ayrıca yükler.
    QR koddan okunan ağaç kimliğini (qr_code) veritabanına kaydeder.
    Ek metrikleri (taç çapı, yüzey alanı, gövde eğikliği vb.) veritabanına kaydeder.
    """
    bucket = storage.bucket() 
    db = firestore.client()

    # --- 1. SPLAT MODEL YÜKLEME VE KALICI LİNK ---
    print(f"[DEBUG] 3D splat model Firebase'e yükleniyor: {splat_model_path}")
    cloud_model_path = f"models/{doc_id}_model.splat"
    out_blob = bucket.blob(cloud_model_path)
    out_blob.upload_from_filename(splat_model_path)
    
    # Özel UUID Token Üretiyoruz
    model_token = str(uuid.uuid4())
    # Token'ı metadata olarak dosyaya işliyoruz
    out_blob.metadata = {"firebaseStorageDownloadTokens": model_token}
    out_blob.patch()
    
    # Kalıcı URL'yi manuel inşa ediyoruz
    encoded_model_path = urllib.parse.quote(cloud_model_path, safe='')
    model_public_url = f"https://firebasestorage.googleapis.com/v0/b/{bucket.name}/o/{encoded_model_path}?alt=media&token={model_token}"

    # --- 1.5. PLY MODEL YÜKLEME VE KALICI LİNK ---
    print(f"[DEBUG] 3D PLY model Firebase'e yükleniyor: {ply_model_path}")
    cloud_ply_path = f"models/{doc_id}_model.ply"
    ply_blob = bucket.blob(cloud_ply_path)
    ply_blob.upload_from_filename(ply_model_path)
    
    # Özel UUID Token Üretiyoruz
    ply_token = str(uuid.uuid4())
    ply_blob.metadata = {"firebaseStorageDownloadTokens": ply_token}
    ply_blob.patch()
    
    encoded_ply_path = urllib.parse.quote(cloud_ply_path, safe='')
    ply_public_url = f"https://firebasestorage.googleapis.com/v0/b/{bucket.name}/o/{encoded_ply_path}?alt=media&token={ply_token}"
    print(f"[SONUÇ] PLY model yüklendi! URL: {ply_public_url}")

    # --- 2. RESİM YÜKLEME VE KALICI LİNK ---
    print(f"[DEBUG] 2D Metrik görüntüsü Firebase'e yükleniyor: {local_image_path}")
    destination_blob_name = f"trees/{doc_id}/metrics_view.jpg"
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(local_image_path)
    
    # Özel UUID Token Üretiyoruz
    image_token = str(uuid.uuid4())
    blob.metadata = {"firebaseStorageDownloadTokens": image_token}
    blob.patch()
    
    encoded_image_path = urllib.parse.quote(destination_blob_name, safe='')
    image_public_url = f"https://firebasestorage.googleapis.com/v0/b/{bucket.name}/o/{encoded_image_path}?alt=media&token={image_token}"

    print(f"[SONUÇ] Dosyalar kalıcı token ile yüklendi! Model URL: {model_public_url}")


    # --- 3. FIRESTORE VERİTABANINI GÜNCELLEME ---
    print(f"[DEBUG] Firestore veritabanı güncelleniyor: {doc_id}")
    print(f"[DEBUG] QR Kod değeri kaydediliyor: {qr_code}")

    # Temel ölçümler
    measurements = {
        "height_m": round(height, 2),
        "diameter_cm": round(diameter, 1),
        "volume_m3": round(volume, 2),
        "fractal_score": round(fractal_score, 3)
    }

    # Ek metrikleri de measurements altına ekle
    if additional_metrics:
        measurements["crown_diameter_m"] = additional_metrics.get("crown_diameter_m", 0.0)
        measurements["crown_surface_area_m2"] = additional_metrics.get("crown_surface_area_m2", 0.0)
        measurements["trunk_lean_deg"] = additional_metrics.get("trunk_lean_deg", 0.0)
        measurements["crown_asymmetry_index"] = additional_metrics.get("crown_asymmetry_index", 0.0)
        measurements["crown_density_pts_m3"] = additional_metrics.get("crown_density_pts_m3", 0.0)
        measurements["crown_base_height_m"] = additional_metrics.get("crown_base_height_m", 0.0)

    tree_data = {
        "status": "completed",
        "qr_code": qr_code,
        "measurements": measurements,
        "transform": {
            "scale": scale_factor,
            "z_offset": float(ground_z) 
        },
        'model_url': model_public_url,
        'ply_url': ply_public_url,
        "metrics_image_url": image_public_url,
        "last_updated": firestore.SERVER_TIMESTAMP
    }

    db.collection("trees").document(doc_id).set(tree_data, merge=True)
    print(f"[SONUÇ] {doc_id} numaralı ağacın tüm dijital ikiz verileri Firebase'e işlendi!")
    print(f"[SONUÇ] QR Kod: {qr_code} | PLY URL: {ply_public_url}")
    if additional_metrics:
        print(f"[SONUÇ] Ek Metrikler: Taç Çapı={additional_metrics['crown_diameter_m']}m, "
              f"Yüzey Alanı={additional_metrics['crown_surface_area_m2']}m², "
              f"Gövde Eğikliği={additional_metrics['trunk_lean_deg']}°, "
              f"Asimetri={additional_metrics['crown_asymmetry_index']}, "
              f"Yoğunluk={additional_metrics['crown_density_pts_m3']} nokta/m³, "
              f"Taç Tabanı={additional_metrics['crown_base_height_m']}m")

    return image_public_url, model_public_url, ply_public_url

In [ ]:
def is_video_h264_mp4(video_path):
    """Videoun H264 codec ve yuv420p piksel formatında olup olmadığını kontrol eder."""
    try:
        cmd = [
            "ffprobe",
            "-v", "error",
            "-select_streams", "v:0",
            "-show_entries", "stream=codec_name,pix_fmt",
            "-of", "json",
            video_path
        ]
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        info = json.loads(result.stdout)

        if "streams" in info and len(info["streams"]) > 0:
            video_stream = info["streams"][0]
            if video_stream.get("codec_name") == "h264" and video_stream.get("pix_fmt") == "yuv420p":
                return True
        return False
    except (subprocess.CalledProcessError, json.JSONDecodeError) as e:
        print(f"[WARNING] ffprobe failed for {video_path}: {e}")
        return False # ffprobe başarısız olursa, formatın istenen gibi olmadığını varsayalım

In [ ]:
def create_2d_metrics_view(pcd, doc_id, height, diameter, volume):
    """
    3D nokta bulutunu kullanarak yan yana (Kuş Bakışı ve Yandan Görünüm)
    iki planlı bir 2D metrik tablosu (Blueprint) oluşturur.
    """
    print(f"[DEBUG] 2D Kapsamlı Metrik Görüntüsü oluşturuluyor: {doc_id}")

    # 1. Nokta Bulutunu Yükle ve Noktaları Al
    points = np.asarray(pcd.points)

    # 2. AYARLAR VE TUVAL BOYUTLARI
    ppm = 150  # Piksel/Metre (1 metre = 150 piksel olacak şekilde ölçekle)
    margin = 1.0  # Kenar boşluğu (metre)

    # Modelin sınırlarını (Bounding Box) bul
    x_min, x_max = np.min(points[:, 0]), np.max(points[:, 0])
    y_min, y_max = np.min(points[:, 1]), np.max(points[:, 1])
    z_min, z_max = np.min(points[:, 2]), np.max(points[:, 2]) # z_min 0'a çok yakındır

    # Alt Tuvallerin Piksel Genişlik ve Yüksekliklerini Hesapla
    # Sol Tuval (Kuş Bakışı - XY)
    top_w = int((x_max - x_min + 2*margin) * ppm)
    top_h = int((y_max - y_min + 2*margin) * ppm)

    # Sağ Tuval (Yandan Görünüm - XZ)
    side_w = int((x_max - x_min + 2*margin) * ppm)
    side_h = int((z_max - z_min + 2*margin) * ppm)

    # Ana Tuval Boyutları (Yan yana iki resim)
    canvas_h = max(top_h, side_h)
    canvas_w = top_w + side_w

    # Beyaz bir ana tuval oluştur (Mimari arka plan)
    metrics_img = np.ones((canvas_h, canvas_w, 3), dtype=np.uint8) * 255

    # Çizgiler için renkler (BGR formatı)
    c_tree = (34, 139, 34)    # Orman Yeşili (Ağaç pikselleri)
    c_measure = (255, 100, 0) # Turuncu (Mimari ölçü okları)
    c_text = (50, 50, 50)     # Koyu Gri (Metinler)
    c_ground = (40, 70, 130)  # Kahverengi Tonu (Toprak çizgisi)

    # -------------------------------------------------------------------
    # 3. SOL TARAF: KUŞ BAKIŞI (TOP VIEW) - Taç Hacmi
    # -------------------------------------------------------------------
    # X ve Y koordinatlarını piksellere çevir
    px_top = ((points[:, 0] - x_min + margin) * ppm).astype(int)
    py_top = ((points[:, 1] - y_min + margin) * ppm).astype(int)

    # Numpy vektörizasyonu ile noktaları tuvale tek hamlede bas
    metrics_img[py_top, px_top] = c_tree

    # Ağacın tam merkezini bul
    cx_top = int((0 - x_min + margin) * ppm)
    cy_top = int((0 - y_min + margin) * ppm)

    # Merkez işaretçisi ve Hacim Bilgisi
    cv2.drawMarker(metrics_img, (cx_top, cy_top), c_measure, markerType=cv2.MARKER_CROSS, markerSize=20, thickness=2)
    cv2.putText(metrics_img, "KUS BAKISI (XY)", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, c_text, 2)
    cv2.putText(metrics_img, f"Tac Hacmi: {volume:.2f} m3", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, c_measure, 2)

    # Ortaya ayıran dikey bir çizgi çek
    cv2.line(metrics_img, (top_w, 0), (top_w, canvas_h), (200, 200, 200), 2)

    # -------------------------------------------------------------------
    # 4. SAĞ TARAF: YANDAN GÖRÜNÜM (SIDE VIEW) - Boy ve Çap
    # -------------------------------------------------------------------
    # X ve Z koordinatlarını piksellere çevir
    px_side = ((points[:, 0] - x_min + margin) * ppm + top_w).astype(int)
    # Görüntülerde Y ekseni aşağı doğru artar, bu yüzden Z eksenini ters çeviriyoruz
    py_side = (canvas_h - (points[:, 2] - z_min + margin) * ppm).astype(int)

    metrics_img[py_side, px_side] = c_tree

    # Zemin (Toprak) Çizgisi
    ground_y = int(canvas_h - (0 - z_min + margin) * ppm)
    cv2.line(metrics_img, (top_w + 20, ground_y), (canvas_w - 20, ground_y), c_ground, 3)
    cv2.putText(metrics_img, "Zemin (Z=0)", (top_w + 20, ground_y + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, c_ground, 2)

    # Merkez X (Sağ tuvaldeki X ekseni ortası)
    cx_side = int((0 - x_min + margin) * ppm + top_w)

    # --- BOY ÖLÇÜSÜ (Dikey Ok) ---
    top_y = int(canvas_h - (height - z_min + margin) * ppm)
    offset_x = 80 # Ağacın biraz yanından çizelim ki karışmasın
    cv2.arrowedLine(metrics_img, (cx_side + offset_x, ground_y), (cx_side + offset_x, top_y), c_measure, 2, tipLength=0.05)
    cv2.arrowedLine(metrics_img, (cx_side + offset_x, top_y), (cx_side + offset_x, ground_y), c_measure, 2, tipLength=0.05)
    cv2.putText(metrics_img, f"Boy: {height:.2f} m", (cx_side + offset_x + 10, int((ground_y + top_y)/2)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, c_measure, 2)

    # --- GÖVDE ÇAPI ÖLÇÜSÜ (Yatay Ok) ---
    # Çapı topraktan yaklaşık 40 cm (0.4m) yukarıdan çiziyoruz
    trunk_y = int(canvas_h - (0.4 - z_min + margin) * ppm)
    radius_px = int((diameter / 200) * ppm) # Çapı yarıçapa ve cm'den metreye çevir

    cv2.arrowedLine(metrics_img, (cx_side - radius_px - 30, trunk_y), (cx_side - radius_px, trunk_y), c_measure, 2)
    cv2.putText(metrics_img, f"Cap: {diameter:.1f} cm", (cx_side - radius_px - 140, trunk_y + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, c_measure, 2)

    cv2.putText(metrics_img, "YANDAN GORUNUM (XZ)", (top_w + 20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, c_text, 2)

    # -------------------------------------------------------------------
    # 5. KAYDET
    # -------------------------------------------------------------------
    # Çıktı klasörünün var olduğundan emin ol
    os.makedirs(f"{WORKSPACE_DIR}/{doc_id}/outputs", exist_ok=True)

    metrics_view_path = f"{WORKSPACE_DIR}/{doc_id}/{doc_id}_metrics.jpg"
    cv2.imwrite(metrics_view_path, metrics_img)
    print(f"[SONUÇ] Mükemmel 2D Mimari Plan kaydedildi: {metrics_view_path}")

    return metrics_view_path

In [ ]:
def process_pending_trees():
    """Firestore'da 'bekliyor' (pending) durumundaki ağaçları bulur ve işler."""
    print("Firebase kontrol ediliyor...")
    pending_docs = db.collection('trees').where('status', '==', 'pending').stream()

    for doc in pending_docs:
        tree_data = doc.to_dict()
        doc_id = doc.id
        video_url_in_storage = tree_data.get('video_url') # PWA'nın storage'a kaydettiği yol

        print(f"Yeni iş bulundu: {doc_id}. İşlem başlatılıyor...")
        print(f"Depolama yolu: {video_url_in_storage}")

        # Durumu güncelliyoruz ki başka bir worker (eğer yatay büyürseniz) bunu almasın
        db.collection('trees').document(doc_id).update({'status': 'processing'})

        try:
            if not video_url_in_storage:
                raise Exception("Video yolu Firebase'de bulunamadı veya boş!")

            # URL'den blob adını çıkar
            parsed_url = urllib.parse.urlparse(video_url_in_storage)
            path_segments = parsed_url.path.split('/')
            # Blob adı '/o/' kısmından sonraki ve '?alt=media' kısmından önceki kısımdır.
            blob_name_encoded = '/'.join(path_segments[path_segments.index('o') + 1:])
            blob_name = urllib.parse.unquote(blob_name_encoded) # URL-decode the blob name

            # 1. Videoyu İndir
            os.makedirs(WORKSPACE_DIR, exist_ok=True)
            local_video_path = os.path.join(WORKSPACE_DIR, f"{doc_id}.mp4")
            blob = bucket.blob(blob_name)
            blob.download_to_filename(local_video_path)
            print("Video indirildi.")

            # YENİ ADIM: OpenCV uyumluluğunu artırmak ve gereksiz yeniden kodlamayı önlemek için kontrol et
            reencoded_video_path = os.path.join(WORKSPACE_DIR, f"{doc_id}_reencoded.mp4")

            if is_video_h264_mp4(local_video_path):
                print(f"[DEBUG] Video {local_video_path} zaten H264 MP4 (yuv420p) formatında. Yeniden kodlama atlanıyor.")
                video_to_analyze = local_video_path
            else:
                print(f"[DEBUG] Videoyu ffmpeg kullanarak {local_video_path} konumundan {reencoded_video_path} konumuna yeniden kodluyor...")
                try:
                    subprocess.run([
                        "ffmpeg", "-y", "-i", local_video_path,
                        "-vcodec", "libx264", "-pix_fmt", "yuv420p",
                        "-preset", "medium", "-crf", "23",
                        reencoded_video_path
                    ], check=True, capture_output=True, text=True)
                    print("[DEBUG] Video yeniden kodlama başarılı.")
                    video_to_analyze = reencoded_video_path
                except subprocess.CalledProcessError as e:
                    print(f"[ERROR] Video yeniden kodlama başarısız oldu: {e.stderr}")
                    video_to_analyze = local_video_path
                except FileNotFoundError:
                    print("[ERROR] ffmpeg komutu bulunamadı. Yeniden kodlama atlanıyor.")
                    video_to_analyze = local_video_path
            print(f"[DEBUG] Analiz edilecek video yolu: {video_to_analyze}")

            # 2. QR Kod Analizi ve Ağaç ID Tespiti
            qr_data, marker_pixel_width, detected_frame_count = analyze_video_for_hybrid_plate(video_to_analyze)

            # QR kod bulunamazsa 'unknown' olarak kaydet
            if qr_data is None:
                print("[UYARI] QR Kod bulunamadı! 'unknown' olarak kaydedilecek.")
                qr_data = "unknown"
            else:
                print(f"[DEBUG] QR Kod başarıyla okundu: {qr_data}")

            # QR kod değerini hemen Firestore'a kaydet (işlem devam ederken bile görünsün)
            db.collection('trees').document(doc_id).update({'qr_code': qr_data})

            if marker_pixel_width is None:
                print("HATA: ArUco Marker videonun başında net olarak bulunamadı!")
            else:
                print(f"Başarılı! Ağaç ID: {qr_data} | Referans Piksel: {marker_pixel_width}")


            # 3. 3D Model Üretimi (COLMAP + 3DGS)
            raw_model_path = run_3d_reconstruction_pipeline(video_to_analyze, doc_id)

            # 4. Ölçeklendirme ve Hacim/Boy Ölçümü (Open3D)
            transforms_path = f"{WORKSPACE_DIR}/{doc_id}/colmap_data/transforms.json"

            scale_factor = calculate_scale_factor(transforms_path, detected_frame_count, marker_pixel_width)
            additional_metrics = None
            if marker_pixel_width is not None:
                
                height, volume, trunk_diameter_cm, fractal_score, ground_z, pcd, additional_metrics = measure_tree_metrics(
                    raw_model_path,
                    scale_factor
                )
                print(f"Ölçümler Tamamlandı: Boy: {height:.2f}m, Hacim: {volume:.2f}m3, Gövde Çapı: {trunk_diameter_cm:.2f}cm, Fraktal Boyut: {fractal_score:.2f}")
                if additional_metrics:
                    print(f"Ek Metrikler: Taç Çapı: {additional_metrics['crown_diameter_m']}m, "
                          f"Taç Yüzey Alanı: {additional_metrics['crown_surface_area_m2']}m², "
                          f"Gövde Eğikliği: {additional_metrics['trunk_lean_deg']}°, "
                          f"Taç Tabanı: {additional_metrics['crown_base_height_m']}m")

                # 5. 3D model 2D metrik imajı oluşturma
                local_image_path = create_2d_metrics_view(pcd, doc_id, height, trunk_diameter_cm, volume)

            # Web için .splat dönüşümünü yap
            splat_model_path = raw_model_path.replace(".ply", ".splat")
            convert_ply_to_splat(raw_model_path, splat_model_path)

            # 6. Firebase'e Geri Yükleme ve Güncelleme (PLY + SPLAT + QR Kod + Ek Metrikler)
            upload_metrics_and_update_db(
                local_image_path=local_image_path,
                splat_model_path=splat_model_path,
                ply_model_path=raw_model_path,
                doc_id=doc_id,
                height=height,
                diameter=trunk_diameter_cm,
                volume=volume,
                fractal_score=fractal_score,
                scale_factor=scale_factor,
                ground_z=ground_z,
                qr_code=qr_data,
                additional_metrics=additional_metrics
            )


        except Exception as e:
            print(f"HATA: {doc_id} işlenirken hata oluştu: {str(e)}")
            db.collection('trees').document(doc_id).update({'status': 'pending', 'error': str(e)})

In [ ]:
# --- CERRAHİ YAMA: PyTorch 3.12 Dynamo Hatasını Fiziksel Olarak Kapat ---
patch_file_path = "/usr/local/lib/python3.12/dist-packages/nerfstudio/utils/misc.py"
if os.path.exists(patch_file_path):
    with open(patch_file_path, "r") as f:
        content = f.read()

    # Hata veren kodu, boş dönen bir lambda fonksiyonuyla değiştir
    old_code = "return torch.compile(*args, **kwargs)"
    new_code = "return lambda x: x"

    if old_code in content:
        content = content.replace(old_code, new_code)
        with open(patch_file_path, "w") as f:
            f.write(content)
        print("[SİSTEM] Nerfstudio Dynamo hatası başarıyla yamalandı!")
# -----------------------------------------------------------------------

# Nerfstudio'nun model yükleme dosyasının yolunu buluyoruz
file_path = '/usr/local/lib/python3.12/dist-packages/nerfstudio/utils/eval_utils.py'

# Dosyayı okuyoruz
with open(file_path, 'r') as file:
    content = file.read()

# PyTorch 2.6'nın güvenlik duvarını aşacak parametreyi koda zorla enjekte ediyoruz
eski_kod = 'torch.load(load_path, map_location="cpu")'
yeni_kod = 'torch.load(load_path, map_location="cpu", weights_only=False)'
content = content.replace(eski_kod, yeni_kod)

# Yamalı dosyayı geri kaydediyoruz
with open(file_path, 'w') as file:
    file.write(content)

print("Nerfstudio, PyTorch 2.6 güvenlik duvarına karşı başarıyla yamalandı!")

In [ ]:
import os
import subprocess

gpu_colmap_path = "/kaggle/working/gpu_colmap"
wrapper_dir = "/kaggle/working/colmap_wrapper"
os.makedirs(wrapper_dir, exist_ok=True)

if not os.path.exists(gpu_colmap_path) or not os.path.exists(f"{gpu_colmap_path}/bin/colmap"):
    print("1. Standalone paket yöneticisi (Micromamba) indiriliyor...")
    subprocess.run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba", shell=True)
    
    print("2. COLMAP indiriliyor (Kısıtlamasız Kök CUDA Versiyonu İsteniyor)...")
    # KRİTİK DEĞİŞİKLİK: cudatoolkit=11.8 KISITLAMASINI SİLDİK! Artık kendi 12.6+ versiyonunu özgürce çekecek.
    subprocess.run(f"./bin/micromamba create -p {gpu_colmap_path} -c conda-forge \"colmap=*=*cuda*\" -y", shell=True)
    
print("\n--- COLMAP GPU Kurulum Bitti. Tercüman (Wrapper) ayarlanıyor... ---")

# TERCÜMAN WRAPPER: Eski Nerfstudio komutlarını Yeni GPU COLMAP diline çevirir
wrapper_script = """#!/bin/bash
ARGS=("$@")
for i in "${!ARGS[@]}"; do
    if [[ "${ARGS[$i]}" == "--SiftExtraction.use_gpu" ]]; then
        ARGS[$i]="--FeatureExtraction.use_gpu"
    elif [[ "${ARGS[$i]}" == "--SiftMatching.use_gpu" ]]; then
        ARGS[$i]="--FeatureMatching.use_gpu"
    fi
done
# Çevrilmiş parametrelerle GERÇEK GPU'LU COLMAP'i çalıştır!
/kaggle/working/gpu_colmap/bin/colmap "${ARGS[@]}"
"""

wrapper_path = f"{wrapper_dir}/colmap"
with open(wrapper_path, "w") as f:
    f.write(wrapper_script)
os.chmod(wrapper_path, 0o755)

# PATH'in en başına sadece tercümanın olduğu klasörü koyuyoruz
current_path = os.environ.get('PATH', '')
clean_path = current_path.replace(f"{gpu_colmap_path}/bin:", "")
os.environ["PATH"] = f"{wrapper_dir}:{clean_path}"

print("--- Tüm sistem başarıyla Tercümana yönlendirildi ve GERÇEK CUDA kütüphaneleri aktif edildi! ---")

In [52]:
# --- İŞÇİYİ BAŞLAT ---
if __name__ == "__main__":
    # Colab defterinde bu hücreyi çalıştırdığınızda bekleyen işleri tarar
    process_pending_trees()

Firebase kontrol ediliyor...
Yeni iş bulundu: mhehdjn1fBIAbRLylWdR. İşlem başlatılıyor...
Depolama yolu: https://firebasestorage.googleapis.com/v0/b/studio-166104040-52130.firebasestorage.app/o/videos%2Fgemini1.mp4?alt=media&token=307a2dc7-3277-41aa-944f-e090ba792653


/usr/local/lib/python3.12/dist-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


Video indirildi.
[DEBUG] Video /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR.mp4 zaten H264 MP4 (yuv420p) formatında. Yeniden kodlama atlanıyor.
[DEBUG] Analiz edilecek video yolu: /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR.mp4
[DEBUG] Hibrit plaka analizi başlıyor: /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR.mp4
[DEBUG] Frame 21'te ArUco Marker Bulundu!
[DEBUG] ArUco Piksel Genişliği: 19.31
HATA: QR Kod veya ArUco Marker videonun başında net olarak bulunamadı!
[mhehdjn1fBIAbRLylWdR] GERÇEK 3D ÜRETİM BAŞLIYOR...
[mhehdjn1fBIAbRLylWdR] Adım 1/3: FFmpeg ve COLMAP çalışıyor (Bu adım videonun uzunluğuna göre 5-15 dk sürer)...
[DEBUG] COLMAP veritabanı /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR/colmap_data/colmap/database.db zaten mevcut. Adım 1 atlanıyor.
[mhehdjn1fBIAbRLylWdR] Adım 2/3: 3DGS modeli eğitiliyor (GPU tam kapasite devrede, 15-25 dk sürer)....
[DEBUG] Eğitilmiş model /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR/ou

2026-03-03 20:58:51.083803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772571531.106411    3293 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772571531.113476    3293 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772571531.132108    3293 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772571531.132137    3293 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772571531.132139    3293 computation_placer.cc:177] computation placer alr

Loading latest checkpoint from load_dir
✅ Done loading checkpoint from outputs/colmap_data/splatfacto/2026-03-03_191814/nerfstudio_models/step-000006999.ckpt
0 Gaussians have NaN/Inf and 7 have low opacity, only export 349824/349831
[mhehdjn1fBIAbRLylWdR] GERÇEK 3D ÜRETİM TAMAMLANDI. Çıktı: /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR/export/splat.ply
[DEBUG] Ölçek Çarpanı (Scale Factor) hesaplanıyor...
[DEBUG] Kameranın ağaca GERÇEK mesafesi: 3.02 metre
UYARI: ArUco bulunan kare transforms.json içinde bulunamadı! Varsayılan çarpan 1.0 kullanılıyor.
[DEBUG] 3D Model analiz ediliyor: /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR/export/splat.ply
[DEBUG] Uygulanan Ölçek Çarpanı: 1.0000
[SONUÇ] Gövde Çapı (30-50cm arası): 2096.0 cm
[DEBUG] Taç apısının Fraktal Skoru hesaplanıyor...
[SONUÇ] Ağacın Fraktal Skoru (D_f): 1.881
[SONUÇ] Ağaç Boyu: 16.62 Metre
[SONUÇ] Taç Hacmi: 7232.65 Metreküp
[SONUÇ] Ağaç Gövde Çapı: 2096.04 Santimetre
[SONUÇ] Ağaç Fraktal Boyutu: 1.88 

In [ ]:
#!rm -r /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR
#!rm -r /kaggle/working/gpu_colmap

In [37]:
#!rm -r /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR/export